# D3-02 Parameterised characterization factors with `edges`

This notebook compares a tighter gasoline-versus-compressed-gas passenger-car pair from the BAFU database.  
The point is to show that `edges` can keep one mapped inventory fixed while characterization factors are reevaluated dynamically from numeric values, symbolic expressions, or external Python functions.


## Learning goals

After this notebook, you should be able to:

- load `edges` method definitions from JSON assets
- parameterise both CH4 and N2O CFs with numeric, symbolic, and external-function rules
- iterate through a CH4 x N2O concentration grid with repeated `evaluate_cfs()` calls
- explain why the matched exchanges stay fixed while CF values and total LCIA scores move


## 1) Reuse a tighter passenger-car pair from the BAFU database


In [ ]:
import json
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D

import bw2data as bd
from edges import EdgeLCIA, SupplyChain

pd.options.display.float_format = '{:,.2f}'.format


### Select the gasoline and compressed-gas passenger-car activities


In [ ]:
bd.projects.set_current('barcelona-rlcia-2026')
bafu = bd.Database('bafu')

gasoline_car = next(
    act for act in bafu
    if act['name'] == 'Transport, passenger car, gasoline, Medium, 2018, EURO-6c'
    and act.get('location') == 'CH'
    and act.get('unit') == 'kilometer'
)

compressed_gas_car = next(
    act for act in bafu
    if act['name'] == 'Transport, passenger car, compressed gas, Medium, 2018, EURO-6c'
    and act.get('location') == 'CH'
    and act.get('unit') == 'kilometer'
)

activities = {
    'Gasoline car (EURO-6c)': gasoline_car,
    'Compressed-gas car (EURO-6c)': compressed_gas_car,
}

pd.DataFrame([
    {
        'activity': label,
        'database': activity.key[0],
        'name': activity['name'],
        'location': activity.get('location'),
        'unit': activity.get('unit'),
        'reference product': activity.get('reference product'),
    }
    for label, activity in activities.items()
])


### Inspect the direct CO2, CH4, N2O and PM 2.5 flows

Both transport datasets emit all four substances directly, but their total climate scores stay close enough to make the CH4 and N2O sensitivity sweep visually interesting.


In [ ]:
target_flows = ['Carbon dioxide, fossil', 'Methane, fossil', 'Dinitrogen monoxide', 'Particulate Matter, < 2.5 um']

flow_rows = []
for label, activity in activities.items():
    direct_flows = {
        exc.input['name']: (exc['amount'], exc.input['unit'])
        for exc in activity.biosphere()
        if exc.input['name'] in target_flows
    }
    for flow_name in target_flows:
        amount, unit = direct_flows.get(flow_name, (0.0, 'kilogram'))
        flow_rows.append(
            {
                'activity': label,
                'supplier name': flow_name,
                'amount': float(amount),
                'unit': unit,
            }
        )

pd.DataFrame(flow_rows).sort_values(['supplier name', 'activity']).style.format({'amount': '{:.6e}'})


### Load the four method definitions from JSON assets


In [ ]:
ASSETS_DIR = Path('tutorials/DAY 3/assets/d3-02')
if not ASSETS_DIR.exists():
    ASSETS_DIR = Path('assets/d3-02')

method_files = {
    'numeric': ASSETS_DIR / 'numeric_climate_method.json',
    'symbolic': ASSETS_DIR / 'symbolic_climate_method.json',
    'external': ASSETS_DIR / 'external_climate_method.json',
    'external_pm_api': ASSETS_DIR / 'external_pm_api_method.json',
}

methods = {
    mode: json.loads(path.read_text())
    for mode, path in method_files.items()
}

pd.DataFrame([
    {
        'mode': mode,
        'file': path.as_posix(),
        'name': methods[mode]['name'],
        'description': methods[mode]['description'],
    }
    for mode, path in method_files.items()
])


## 2) Build a documented two-gas CF model

We keep the time horizon fixed at 100 years. The next cells do three things in sequence: define the baseline atmospheric state, collect the physical constants used by the GWP equations, and wrap those equations in small helper functions that can be reused by the JSON methods. The external-function method reflects a gas-specific radiative efficiency that decreases with `1 / sqrt(C)`, while the symbolic method keeps only that concentration term and freezes the remaining factors at the reference state.


### 2.1 Reference concentrations and time horizon


In [ ]:
CH4_REFERENCE_PPB = 1866.0
N2O_REFERENCE_PPB = 332.0
H_REFERENCE_YEARS = 100.0

pd.DataFrame([
    {'gas': 'CH4', 'reference concentration [ppb]': CH4_REFERENCE_PPB, 'time horizon [years]': H_REFERENCE_YEARS},
    {'gas': 'N2O', 'reference concentration [ppb]': N2O_REFERENCE_PPB, 'time horizon [years]': H_REFERENCE_YEARS},
])


### 2.2 Physical constants from the `edges` GWP example

These values are reused directly from the `edges` publication-style GWP example. The goal here is not to re-derive the climate model from scratch, but to expose the quantities that control how CH4 and N2O characterization factors change when atmospheric concentrations move away from the reference state.


In [ ]:
# Total atmospheric mass [kg] and mean molar mass of air [g/mol].
# Together they let us convert a ppb-based radiative-forcing term to a mass-based term.
M_ATM = 5.15e18
M_AIR = 28.96

# Molecular masses [g/mol] for the gases characterized in this notebook.
M_GAS = {
    'CO2': 44.01,
    'CH4': 16.04,
    'N2O': 44.013,
}

# Radiative-efficiency coefficients. CH4 carries an additional indirect-forcing multiplier.
RF_COEFF = {
    'CH4': 0.036,
    'N2O': 0.12,
}

INDIRECT_FACTOR = {
    'CH4': 1.65,
    'N2O': 1.0,
}

# Atmospheric lifetimes [years] for the gas-specific perturbation response.
TAU_GAS = {
    'CH4': 11.8,
    'N2O': 109.0,
}

# Multi-exponential impulse-response model for the CO2 reference system.
CO2_IRF = {
    'a0': 0.2173,
    'a': [0.2240, 0.2824, 0.2763],
    'tau': [394.4, 36.54, 4.304],
}


### 2.3 Functions for concentration-dependent GWP

The helper functions follow the same logic as the publication example: convert radiative-efficiency terms from concentration units to mass-based units, integrate the CO2 reference response over the selected horizon, then compute gas-specific AGWP values and divide by the CO2 AGWP to obtain a GWP.


In [ ]:
def convert_ppb_to_mass_rf(a_ppb, gas):
    """Convert a ppb-scale radiative-forcing term into a mass-based term for one gas."""
    return a_ppb * (M_ATM / M_GAS[gas]) * (M_AIR / 1e9)


def agwp_co2(horizon_years):
    """Compute the CO2 reference AGWP over the chosen time horizon."""
    integral = CO2_IRF['a0'] * horizon_years + sum(
        a * tau * (1 - np.exp(-horizon_years / tau))
        for a, tau in zip(CO2_IRF['a'], CO2_IRF['tau'])
    )
    return convert_ppb_to_mass_rf(1.37e-5, 'CO2') * integral


In [ ]:
def radiative_efficiency_concentration(gas, concentration_ppb):
    """Apply the concentration-dependent 1 / sqrt(C) radiative-efficiency term."""
    alpha = RF_COEFF[gas]
    return (alpha / (2 * np.sqrt(concentration_ppb))) * INDIRECT_FACTOR[gas]


def agwp_gas(gas, horizon_years, concentration_ppb):
    """Integrate the gas-specific radiative-forcing response over the selected horizon."""
    mass_based_rf = convert_ppb_to_mass_rf(
        radiative_efficiency_concentration(gas, concentration_ppb),
        gas,
    )
    tau = TAU_GAS[gas]
    return mass_based_rf * tau * (1 - np.exp(-horizon_years / tau))


def gwp_concentration_dependent(gas, horizon_years, concentration_ppb):
    """Return the concentration-dependent GWP by dividing AGWP_gas by AGWP_CO2."""
    return agwp_gas(gas, horizon_years, concentration_ppb) / agwp_co2(horizon_years)


### 2.4 Reference CFs at the baseline atmospheric state


In [ ]:
# Fixed benchmark CFs at the baseline atmospheric state.
# The numeric method uses these values directly, and the symbolic method rescales them.
REFERENCE_CF = {
    'CH4': gwp_concentration_dependent('CH4', H_REFERENCE_YEARS, CH4_REFERENCE_PPB),
    'N2O': gwp_concentration_dependent('N2O', H_REFERENCE_YEARS, N2O_REFERENCE_PPB),
}

pd.DataFrame([
    {
        'gas': gas,
        'reference concentration [ppb]': CH4_REFERENCE_PPB if gas == 'CH4' else N2O_REFERENCE_PPB,
        'reference CF [kg CO2-eq / kg gas]': value,
    }
    for gas, value in REFERENCE_CF.items()
])


### 2.5 Build a CH4 x N2O concentration grid


In [ ]:
# Sweep low-to-high CH4 and N2O concentrations around the baseline atmosphere.
ch4_levels = [1200.0, 1500.0, CH4_REFERENCE_PPB, 2200.0, 2600.0]
n2o_levels = [280.0, 310.0, N2O_REFERENCE_PPB, 360.0, 390.0]

# Each row is one atmospheric state that will be passed to EdgeLCIA as a scenario index.
parameter_grid = pd.DataFrame([
    {
        'case': f'case_{index:02d}',
        'ch4_concentration_ppb': ch4,
        'n2o_concentration_ppb': n2o,
    }
    for index, (ch4, n2o) in enumerate(product(ch4_levels, n2o_levels), start=1)
]).set_index('case')

# Package the grid in the parameter structure expected by the JSON methods.
grid_parameters = {
    'concentration grid': {
        'ch4_concentration_ppb': parameter_grid['ch4_concentration_ppb'].to_dict(),
        'n2o_concentration_ppb': parameter_grid['n2o_concentration_ppb'].to_dict(),
        'ch4_reference_ppb': CH4_REFERENCE_PPB,
        'n2o_reference_ppb': N2O_REFERENCE_PPB,
        'ch4_gwp100_reference': REFERENCE_CF['CH4'],
        'n2o_gwp100_reference': REFERENCE_CF['N2O'],
        'horizon_years': H_REFERENCE_YEARS,
    }
}

parameter_grid.reset_index()


The first three JSON methods all target the same greenhouse-gas flows, but they inject the CFs at different stages. The numeric method uses the fixed reference values above, the symbolic method rescales those reference values analytically with the `1 / sqrt(C)` term, and the external climate method calls the Python function directly so the full concentration-dependent expression is reevaluated for every case in the grid. The fourth JSON asset is used later for a separate non-climate example that calls a live web API.


## 3) Use one simple reevaluation pattern

Instead of hiding the workflow in helper functions, the next sections write out the same `EdgeLCIA` steps explicitly: build the object, run `lci()`, map the exchanges once, then reevaluate CFs and call `lcia()` for each case.


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plot_style_map = {
    'Gasoline car (EURO-6c)': {'color': '#355070', 'marker': 'o'},
    'Compressed-gas car (EURO-6c)': {'color': '#E56B6F', 'marker': '^'},
}



In [ ]:
# Keep the vertical axis fixed across the two surface plots so they are easy to compare.
plot_z_limits = (0.2, 0.3)

We now repeat the same explicit pattern in each section: build an `EdgeLCIA` object, run `lci()` once, map the exchanges once, then reevaluate CFs and run `lcia()` for each atmospheric case.


The 3D plotting cells below also stay explicit: they reshape the result table with `pivot(...)` and draw both surfaces directly in one shared set of axes.


## 4) Start with the fixed numeric benchmark


In [ ]:
numeric_rows = []

for label, activity in activities.items():
    print(label)
    lca = EdgeLCIA(demand={activity: 1}, method=methods['numeric'])
    lca.lci()
    lca.map_exchanges()
    lca.evaluate_cfs()
    lca.lcia()

    cf_table = lca.generate_cf_table(include_unmatched=False).copy()
    methane_cf = np.nan
    n2o_cf = np.nan

    if not cf_table.empty and 'supplier name' in cf_table.columns and 'CF' in cf_table.columns:
        for _, row in cf_table.iterrows():
            supplier_name = '' if pd.isna(row['supplier name']) else str(row['supplier name'])
            if supplier_name.startswith('Methane') and pd.notna(row['CF']):
                methane_cf = float(row['CF'])
            if supplier_name.startswith('Dinitrogen monoxide') and pd.notna(row['CF']):
                n2o_cf = float(row['CF'])

    numeric_rows.append(
        {
            'activity': label,
            'methane CF': methane_cf,
            'n2o CF': n2o_cf,
            'score': float(lca.score),
        }
    )

numeric_summary = pd.DataFrame(numeric_rows)
numeric_summary



The fixed benchmark pins both gases to one reference atmospheric state.  
The inventory and the matched exchanges are already in place; the later steps only change the CF values.


## 5) Sweep the symbolic CH4 and N2O approximation


In [ ]:
symbolic_rows = []

for label, activity in activities.items():
    print(label)
    lca = EdgeLCIA(
        demand={activity: 1},
        method=methods['symbolic'],
        parameters=grid_parameters,
        scenario='concentration grid',
    )
    lca.lci()
    lca.map_exchanges()

    for case, values in parameter_grid.iterrows():
        lca.evaluate_cfs(scenario_idx=case)
        lca.lcia()

        symbolic_rows.append(
            {
                'case': case,
                'ch4_concentration_ppb': values['ch4_concentration_ppb'],
                'n2o_concentration_ppb': values['n2o_concentration_ppb'],
                'activity': label,
                'score': float(lca.score),
            }
        )

symbolic_results = pd.DataFrame(symbolic_rows)
symbolic_results.head(8)



In [ ]:
results = symbolic_results
title = 'Symbolic reevaluation of CH4 and N2O CFs'

fig = plt.figure(figsize=(9.2, 6.8))
ax = fig.add_subplot(111, projection='3d')

z_floor = float(results['score'].min())
z_ceiling = float(results['score'].max())
z_padding = max((z_ceiling - z_floor) * 0.08, 0.2)
z_floor -= z_padding
z_ceiling += z_padding

for label, style in plot_style_map.items():
    subset = results.loc[results['activity'] == label].copy()
    surface = (
        subset.pivot(index='n2o_concentration_ppb', columns='ch4_concentration_ppb', values='score')
        .sort_index()
        .sort_index(axis=1)
    )

    x_values = surface.columns.to_numpy(dtype=float)
    y_values = surface.index.to_numpy(dtype=float)
    X, Y = np.meshgrid(x_values, y_values)
    Z = surface.to_numpy(dtype=float)

    ax.plot_surface(
        X, Y, Z,
        color=style['color'],
        alpha=0.62,
        linewidth=0.45,
        edgecolor=style['color'],
        antialiased=True,
        shade=False,
    )
    ax.scatter(
        subset['ch4_concentration_ppb'],
        subset['n2o_concentration_ppb'],
        subset['score'],
        color=style['color'],
        marker=style['marker'],
        s=28,
        edgecolors='black',
        linewidths=0.35,
        depthshade=False,
    )
    ax.contour(
        X, Y, Z,
        zdir='z',
        offset=z_floor,
        levels=6,
        colors=[style['color']],
        linewidths=1.1,
        alpha=0.9,
    )

ax.set_title(title, pad=14)
ax.set_xlabel('CH4 concentration [ppb]')
ax.set_ylabel('N2O concentration [ppb]')
ax.set_zlabel('LCIA score [kg CO2-eq / km]')
ax.set_zlim(*plot_z_limits)
ax.view_init(elev=24, azim=-132)
ax.xaxis.pane.set_alpha(0.06)
ax.yaxis.pane.set_alpha(0.06)
ax.zaxis.pane.set_alpha(0.0)
ax.grid(True, alpha=0.22)

legend_handles = [
    Patch(facecolor=plot_style_map[label]['color'], edgecolor=plot_style_map[label]['color'], alpha=0.62, label=label)
    for label in plot_style_map
] + [
    Line2D([0], [0], color=plot_style_map[label]['color'], marker=plot_style_map[label]['marker'], linestyle='',
           markeredgecolor='black', markeredgewidth=0.35, label=f'{label} grid points')
    for label in plot_style_map
]
ax.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(0.02, 0.98), frameon=False)
plt.show()



## Checkpoint 1

Use the symbolic grid to identify, for each car technology, which concentration pair gives the lowest score and which gives the highest score.


In [ ]:
# TODO
# Start from `symbolic_results` and summarise the score extremes for both activities.


## 6) Sweep the external concentration-dependent function


In [ ]:
allowed_functions = {
    'gwp_concentration_dependent': gwp_concentration_dependent,
}
external_rows = []


for label, activity in activities.items():
    print(label)
    lca = EdgeLCIA(
        demand={activity: 1},
        method=methods['external'],
        parameters=grid_parameters,
        scenario='concentration grid',
        allowed_functions=allowed_functions,
    )
    lca.lci()
    lca.map_exchanges()

    for case, values in parameter_grid.iterrows():
        lca.evaluate_cfs(scenario_idx=case)
        lca.lcia()
        
        external_rows.append(
            {
                'case': case,
                'ch4_concentration_ppb': values['ch4_concentration_ppb'],
                'n2o_concentration_ppb': values['n2o_concentration_ppb'],
                'activity': label,
                'score': float(lca.score),
            }
        )

external_results = pd.DataFrame(external_rows)
external_results.head(8)

In [ ]:
results = external_results
title = 'External-function reevaluation of CH4 and N2O CFs'

fig = plt.figure(figsize=(9.2, 6.8))
ax = fig.add_subplot(111, projection='3d')

z_floor = float(results['score'].min())
z_ceiling = float(results['score'].max())
z_padding = max((z_ceiling - z_floor) * 0.08, 0.2)
z_floor -= z_padding
z_ceiling += z_padding

for label, style in plot_style_map.items():
    subset = results.loc[results['activity'] == label].copy()
    surface = (
        subset.pivot(index='n2o_concentration_ppb', columns='ch4_concentration_ppb', values='score')
        .sort_index()
        .sort_index(axis=1)
    )

    x_values = surface.columns.to_numpy(dtype=float)
    y_values = surface.index.to_numpy(dtype=float)
    X, Y = np.meshgrid(x_values, y_values)
    Z = surface.to_numpy(dtype=float)

    ax.plot_surface(
        X, Y, Z,
        color=style['color'],
        alpha=0.62,
        linewidth=0.45,
        edgecolor=style['color'],
        antialiased=True,
        shade=False,
    )
    ax.scatter(
        subset['ch4_concentration_ppb'],
        subset['n2o_concentration_ppb'],
        subset['score'],
        color=style['color'],
        marker=style['marker'],
        s=28,
        edgecolors='black',
        linewidths=0.35,
        depthshade=False,
    )
    ax.contour(
        X, Y, Z,
        zdir='z',
        offset=z_floor,
        levels=6,
        colors=[style['color']],
        linewidths=1.1,
        alpha=0.9,
    )

ax.set_title(title, pad=14)
ax.set_xlabel('CH4 concentration [ppb]')
ax.set_ylabel('N2O concentration [ppb]')
ax.set_zlabel('LCIA score [kg CO2-eq / km]')
ax.set_zlim(*plot_z_limits)
ax.view_init(elev=24, azim=-132)
ax.xaxis.pane.set_alpha(0.06)
ax.yaxis.pane.set_alpha(0.06)
ax.zaxis.pane.set_alpha(0.0)
ax.grid(True, alpha=0.22)

legend_handles = [
    Patch(facecolor=plot_style_map[label]['color'], edgecolor=plot_style_map[label]['color'], alpha=0.62, label=label)
    for label in plot_style_map
] + [
    Line2D([0], [0], color=plot_style_map[label]['color'], marker=plot_style_map[label]['marker'], linestyle='',
           markeredgecolor='black', markeredgewidth=0.35, label=f'{label} grid points')
    for label in plot_style_map
]
ax.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(0.02, 0.98), frameon=False)
plt.show()



### Compare the score ranges across the three modes


In [ ]:
mode_rows = []
for mode, results in [
    ('Numeric benchmark', numeric_summary),
    ('Symbolic grid', symbolic_results),
    ('External grid', external_results),
]:
    row = {'mode': mode}
    for label in activities:
        scores = results.loc[results['activity'] == label, 'score']
        row[f'{label} min score'] = float(scores.min())
        row[f'{label} max score'] = float(scores.max())
    mode_rows.append(row)

pd.DataFrame(mode_rows)


## 7) Call an external API for location-specific PM2.5 CFs

The previous sections changed the same CFs for all edges at once. This last example does something different: it uses one country-specific row per consumer location in the JSON method file, so the CF for `Particulate Matter, < 2.5 um` depends on **where the emission is matched in the supply chain**.

This is still a teaching example, not a validated LCIA model. We use the current PM2.5 background concentration from Open-Meteo together with a documented intake-fraction archetype and a background-dependent inhaled effect factor. Because the API returns live data, participants will not necessarily obtain identical values.

The teaching model is deliberately simple:

$$
\mathrm{CF}_{\mathrm{PM2.5}}(c, e) = iF(e) \times EF\big(C(c)\big)
$$

where:

- $c$ is the country where the matched emission takes place,
- $e$ is the emission environment (`urban`, `rural`, or `remote`),
- $C(c)$ is the current background PM2.5 concentration from Open-Meteo,
- $iF(e)$ is the intake fraction for the chosen environment,
- $EF(C)$ is the background-dependent effect factor.

The effect factor is scaled from a reference value with:

$$
EF(C) = EF_{ref} \times \frac{1 + \beta C_{ref}}{1 + \beta C}
$$

and

$$
\beta = RR_{1\,\mu g}^{\vphantom{1}} - 1 = RR_{10\,\mu g}^{1/10} - 1
$$

This time the JSON method also uses the elementary-flow `categories` field to choose the intake-fraction archetype:

- `('air', 'urban air close to ground')` -> `urban`
- `('air', 'non-urban air or from high stacks')` -> `rural`
- `('air', 'low population density, long-term')` -> `remote`
- `('air', 'lower stratosphere + upper troposphere')` -> `remote`
- generic `('air',)` -> `rural` as a fallback

The method file already includes `map_aggregate_locations`, `map_dynamic_locations`, `map_contained_locations`, and `map_remaining_locations_to_global`, so aggregate consumer locations such as `RER` can still be resolved from the country-specific rows.

This section needs internet access. The first cell below can take a bit of time, because it fills a cache for all country rows in the method before `edges` starts evaluating CFs.

In [ ]:
import requests

# Read the centroid table from the JSON asset we created for this tutorial.
# Each row gives us a country ISO2 code, a country name, and a latitude/longitude pair.
COUNTRY_CENTROIDS = {
    row['code']: row
    for row in json.loads((ASSETS_DIR / 'country_centroids_iso2.json').read_text())
}

# Open-Meteo endpoint used to fetch the *current* background PM2.5 concentration.
OPEN_METEO_URL = 'https://air-quality-api.open-meteo.com/v1/air-quality'

# Small caches so we don't call the API or recompute the same CF over and over.
PM25_BACKGROUND_CACHE = {}
PM25_CF_CACHE = {}

# Humbert et al. archetypal outdoor intake fractions for primary PM2.5.
# Units: kg inhaled / kg emitted.
# The urban value is much larger because more people are exposed nearby.
PM25_INTAKE_FRACTIONS = {
    'urban': 26e-6,
    'rural': 2.6e-6,
    'remote': 0.1e-6,
}

# Map elementary-flow air subcompartments to the simple teaching archetypes.
# This is the bridge between the biosphere flow metadata and the PM CF function.
PM25_CATEGORY_ENVIRONMENTS = {
    ('air', 'urban air close to ground'): 'urban',
    ('air', 'non-urban air or from high stacks'): 'rural',
    ('air', 'low population density, long-term'): 'remote',
    ('air', 'lower stratosphere + upper troposphere'): 'remote',
    ('air',): 'rural',
}


def _fetch_background_pm25(country_code):
    # Convert the code to upper case so 'ch' and 'CH' behave the same way.
    code = country_code.upper()

    # Stop early if we have no centroid for this country.
    if code not in COUNTRY_CENTROIDS:
        raise ValueError(f"No centroid available for country code '{country_code}'")

    # Reuse a previously downloaded concentration if we already have it.
    if code in PM25_BACKGROUND_CACHE:
        return PM25_BACKGROUND_CACHE[code]

    # The API only needs latitude, longitude, and the variable we want.
    params = {
        'latitude': COUNTRY_CENTROIDS[code]['lat'],
        'longitude': COUNTRY_CENTROIDS[code]['lon'],
        'current': 'pm2_5',
    }

    try:
        # Ask the external API for the current PM2.5 concentration.
        response = requests.get(OPEN_METEO_URL, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        background = float(data['current']['pm2_5'])
    except Exception as exc:
        raise RuntimeError(
            f"The Open-Meteo call failed for {code}. This section needs internet access and a valid API response."
        ) from exc

    # Save the result in the cache before returning it.
    PM25_BACKGROUND_CACHE[code] = background
    return background


def _beta_from_rr10(rr_per_10ug=1.08):
    # We use a relative risk per 10 ug/m3 and convert it to an approximate
    # per-ug/m3 slope. This gives the beta used in the simple EF equation.
    if rr_per_10ug <= 1:
        raise ValueError('rr_per_10ug must be > 1')
    return rr_per_10ug ** (1 / 10) - 1


def _effect_factor_pm25_daly_per_kg_inhaled(
    background_ug_m3,
    ef_ref_daly_per_kg_inhaled=54.0,
    ref_background_ug_m3=20.0,
    rr_per_10ug=1.08,
):
    # A few simple guards make mistakes easier to diagnose.
    if background_ug_m3 < 0:
        raise ValueError('background_ug_m3 must be >= 0')
    if ref_background_ug_m3 < 0:
        raise ValueError('ref_background_ug_m3 must be >= 0')
    if ef_ref_daly_per_kg_inhaled <= 0:
        raise ValueError('ef_ref_daly_per_kg_inhaled must be > 0')

    # Convert the RR into beta, then scale the reference effect factor.
    beta = _beta_from_rr10(rr_per_10ug)
    scale = (1 + beta * ref_background_ug_m3) / (1 + beta * background_ug_m3)
    return ef_ref_daly_per_kg_inhaled * scale


def evaluate_pm25_cf(
    country_code,
    environment='rural',
    ef_ref_daly_per_kg_inhaled=54.0,
    ref_background_ug_m3=20.0,
    rr_per_10ug=1.08,
):
    # This is the function that the JSON method calls from edges.
    # It returns a full dictionary so we can inspect intermediate values,
    # not just the final scalar CF.
    code = country_code.upper()
    env = environment.lower()

    # Only allow the three archetypes we defined above.
    if env not in PM25_INTAKE_FRACTIONS:
        raise ValueError(f"environment must be one of {tuple(PM25_INTAKE_FRACTIONS)}")

    # Cache the complete result so repeated calls for the same country and
    # environment do not hit the API again.
    cache_key = (code, env, ef_ref_daly_per_kg_inhaled, ref_background_ug_m3, rr_per_10ug)
    if cache_key in PM25_CF_CACHE:
        return PM25_CF_CACHE[cache_key]

    # Step 1: get the live background concentration from Open-Meteo.
    background = _fetch_background_pm25(code)

    # Step 2: choose the intake fraction from the environment archetype.
    intake_fraction = PM25_INTAKE_FRACTIONS[env]

    # Step 3: compute the inhaled effect factor at the current background level.
    effect_factor = _effect_factor_pm25_daly_per_kg_inhaled(
        background_ug_m3=background,
        ef_ref_daly_per_kg_inhaled=ef_ref_daly_per_kg_inhaled,
        ref_background_ug_m3=ref_background_ug_m3,
        rr_per_10ug=rr_per_10ug,
    )

    # Step 4: combine intake fraction and effect factor into the final CF.
    cf = intake_fraction * effect_factor

    # Keep the intermediate values for teaching and debugging.
    result = {
        'country': code,
        'country_name': COUNTRY_CENTROIDS[code]['name'],
        'pollutant': 'pm2_5',
        'environment': env,
        'background_concentration_ug_m3': background,
        'intake_fraction_kg_inhaled_per_kg_emitted': intake_fraction,
        'effect_factor_daly_per_kg_inhaled': effect_factor,
        'cf_daly_per_kg_emitted': cf,
        'cf_unit': 'DALY / kg PM2.5 emitted',
    }

    PM25_CF_CACHE[cache_key] = result
    return result

### Fill the country cache once and inspect the decomposition

The JSON method file contains one row per country location. `edges` will reevaluate all those expressions, so we first fill a cache once. Then the later `apply_strategies()` and `evaluate_cfs()` calls can reuse the cached values instead of making repeated HTTP requests.


In [ ]:
all_country_codes = sorted(COUNTRY_CENTROIDS)

# Warm up the cache once for all countries that appear in the method file.
# This makes the later edges run faster and also shows participants that the
# method is really backed by live country-specific values.
for index, code in enumerate(all_country_codes, start=1):
    evaluate_pm25_cf(code, 'rural')
    if index % 25 == 0 or index == len(all_country_codes):
        print(f'Fetched {index} / {len(all_country_codes)} country-specific PM2.5 backgrounds')

# Show the mapping from biosphere subcompartment to the teaching environment.
pm25_category_mapping = pd.DataFrame([
    {
        'supplier categories': ' / '.join(categories),
        'environment': environment,
        'intake fraction [kg inhaled / kg emitted]': PM25_INTAKE_FRACTIONS[environment],
    }
    for categories, environment in PM25_CATEGORY_ENVIRONMENTS.items()
])
display(pm25_category_mapping.style.format({'intake fraction [kg inhaled / kg emitted]': '{:.3e}'}))

# Build a larger reference table for 20 countries and all three environments.
# This gives a wider view of how the live country background values interact
# with the urban / rural / remote intake-fraction archetypes.
reference_country_codes = [
    'CH', 'DE', 'FR', 'IT', 'ES',
    'GB', 'US', 'CN', 'IN', 'BR',
    'ZA', 'AU', 'SE', 'NO', 'JP',
    'MX', 'CA', 'TR', 'ID', 'EG',
]

sample_country_rows = []
for code in reference_country_codes:
    for environment in ['urban', 'rural', 'remote']:
        pm_result = evaluate_pm25_cf(code, environment)
        sample_country_rows.append(
            {
                'country_code': code,
                'country_name': pm_result['country_name'],
                'environment': pm_result['environment'],
                'background PM2.5 [ug/m3]': pm_result['background_concentration_ug_m3'],
                'intake fraction [kg inhaled / kg emitted]': pm_result['intake_fraction_kg_inhaled_per_kg_emitted'],
                'effect factor [DALY / kg inhaled]': pm_result['effect_factor_daly_per_kg_inhaled'],
                'CF [DALY / kg emitted]': pm_result['cf_daly_per_kg_emitted'],
            }
        )

sample_country_reference = pd.DataFrame(sample_country_rows)
sample_country_reference.style.format({
    'background PM2.5 [ug/m3]': '{:.2f}',
    'intake fraction [kg inhaled / kg emitted]': '{:.3e}',
    'effect factor [DALY / kg inhaled]': '{:.3f}',
    'CF [DALY / kg emitted]': '{:.3e}',
})

### Run the location-specific method through the full supply chain

The method file now has one `Particulate Matter, < 2.5 um` row per country **and per PM2.5 air subcompartment**. In the original BAFU naming this flow corresponds to `Particulates, < 2.5 um`, but after import the biosphere flow is matched to the standard name used in the Brightway biosphere database. We now let `apply_strategies()` run the full location cascade so `edges` can assign country-specific PM2.5 CFs to the actual matched consumer locations in the supply chain.

Because the supplier `categories` are part of the method rows, the same country can receive different CFs for `urban`, `rural`, or `remote` PM2.5 emissions depending on the air compartment of the matched biosphere flow.

In [ ]:
pm_score_rows = []
pm_location_rows = []

# Run the external PM method for both vehicle systems.
for label, activity in activities.items():
    print(label)
    lca = EdgeLCIA(
        demand={activity: 1},
        method=methods['external_pm_api'],
        allowed_functions={'evaluate_pm25_cf': evaluate_pm25_cf},
    )
    lca.lci()
    lca.apply_strategies()
    lca.evaluate_cfs()
    lca.lcia()

    # Store the total PM score for this activity.
    pm_score_rows.append(
        {
            'activity': label,
            'score': float(lca.score),
            'unit': methods['external_pm_api']['unit'],
        }
    )

    # Extract the characterized rows so we can inspect where the PM score comes from.
    cf_table = lca.generate_cf_table(include_unmatched=False).copy()
    if not cf_table.empty and 'supplier name' in cf_table.columns:
        for _, row in cf_table.iterrows():
            supplier_name = '' if pd.isna(row['supplier name']) else str(row['supplier name'])

            # Keep only the PM2.5 rows for this example.
            if supplier_name.startswith('Particulate Matter, < 2.5 um'):
                categories_value = row['supplier categories'] if 'supplier categories' in row else None

                # Convert the categories back to a tuple so we can use the mapping dictionary.
                if isinstance(categories_value, list):
                    categories_key = tuple(categories_value)
                elif isinstance(categories_value, tuple):
                    categories_key = categories_value
                else:
                    categories_key = ('air',)

                pm_location_rows.append(
                    {
                        'activity': label,
                        'consumer location': row['consumer location'],
                        'supplier categories': ' / '.join(categories_key),
                        'environment': PM25_CATEGORY_ENVIRONMENTS.get(categories_key, 'rural'),
                        'CF': float(row['CF']),
                        'impact': float(row['impact']),
                    }
                )

pm_score_summary = pd.DataFrame(pm_score_rows)
pm_location_results = pd.DataFrame(pm_location_rows)
pm_score_summary.style.format({'score': '{:.3e}'})

### Inspect the matched supply-chain locations

This summary shows which consumer locations contributed to the PM2.5-emission score, together with the location-specific CF that was applied there. This is where you can check whether aggregate locations such as `RER` were resolved into country-based fallback CFs.


In [ ]:
# Aggregate the PM results by activity, matched consumer location, and
# environment archetype. The mean CF is enough here because rows with the
# same location/environment can appear multiple times in the supply chain.
pm_location_summary = (
    pm_location_results.groupby(['activity', 'consumer location', 'environment'], as_index=False)
    .agg({'CF': 'mean', 'impact': 'sum'})
    .sort_values(['activity', 'impact'], ascending=[True, False])
)
pm_location_summary.style.format({'CF': '{:.3e}', 'impact': '{:.3e}'})

### Plot a few live country CFs and the location contributions

The left panel shows a small subset of the live country-specific PM2.5 CFs. The right panel shows how the PM2.5 score is distributed across matched supply-chain locations for the two vehicle options.


In [ ]:
# Keep the main consumer locations and pool the rest into 'Other' so the
# stacked bar plot stays readable.
top_locations = (
    pm_location_summary.groupby('consumer location')['impact']
    .sum()
    .sort_values(ascending=False)
    .head(8)
    .index
)

plot_location_rows = []
for _, row in pm_location_summary.iterrows():
    location_label = row['consumer location'] if row['consumer location'] in top_locations else 'Other'
    plot_location_rows.append(
        {
            'activity': row['activity'],
            'location': location_label,
            'impact': row['impact'],
        }
    )

# Prepare a pivot table for the stacked bar plot of location contributions.
plot_location_table = (
    pd.DataFrame(plot_location_rows)
    .groupby(['activity', 'location'], as_index=False)['impact']
    .sum()
)
plot_location_pivot = (
    plot_location_table.pivot(index='activity', columns='location', values='impact')
    .fillna(0)
)

# Prepare a second pivot table to compare urban / rural / remote CFs for 20 countries.
plot_country_reference = (
    sample_country_reference.pivot(index='country_name', columns='environment', values='CF [DALY / kg emitted]')
    .loc[[COUNTRY_CENTROIDS[code]['name'] for code in reference_country_codes]]
)

fig, axes = plt.subplots(1, 2, figsize=(16.5, 4.8), constrained_layout=True)

# Left plot: the same country can have three different CFs depending on the
# PM2.5 subcompartment because the intake fraction changes.
plot_country_reference.plot(
    kind='bar',
    ax=axes[0],
    width=0.78,
    color=['#d62828', '#457b9d', '#6c757d'],
)
axes[0].set_title('Live PM2.5 CFs by country and air subcompartment')
axes[0].set_ylabel('CF [DALY / kg emitted]')
axes[0].tick_params(axis='x', rotation=55)
axes[0].ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
axes[0].legend(title='Environment', frameon=False)

# Right plot: supply-chain contribution by matched consumer location.
plot_location_pivot.plot(
    kind='bar',
    stacked=True,
    ax=axes[1],
    width=0.7,
    colormap='tab20',
)
axes[1].set_title('Location contributions to the PM2.5 score')
axes[1].set_ylabel('Impact [DALY / km]')
axes[1].ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
axes[1].legend(title='Consumer location', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
axes[1].tick_params(axis='x', rotation=0)

plt.show()

This last section illustrates a different kind of variability. The greenhouse-gas example changed a small parameter grid explicitly. Here the CF depends on a live service and on the matched supply-chain location, and it is decomposed into an intake fraction and a background-dependent inhaled effect factor.


### Investigate the Brazil contribution with a Sankey diagram

The location summary can tell us that `BR` matters for the gasoline-car PM score, but it does not show **which supply-chain branch** is responsible. To inspect that, we can reuse `edges`' `SupplyChain` helper with the same external PM method and render a Sankey diagram for the gasoline-car activity only.

The next cell does three things:

- runs a deeper supply-chain traversal for the gasoline car,
- prints the rows whose matched location is `BR`,
- saves and displays an interactive Sankey diagram so you can follow the Brazilian branch visually.

In [ ]:
gasoline_pm_supply_chain = SupplyChain(
    activity=gasoline_car,
    method=methods['external_pm_api'],
    amount=1,
    level=6,
    cutoff=0.01,
    cutoff_basis='total',
)

# SupplyChain creates its own EdgeLCIA object internally, so we need to add
# the external PM function to that object's safe globals before bootstrapping.
gasoline_pm_supply_chain.elcia.SAFE_GLOBALS.update({'evaluate_pm25_cf': evaluate_pm25_cf})

gasoline_pm_total_score = gasoline_pm_supply_chain.bootstrap()
gasoline_pm_sankey_df, gasoline_pm_total_score, gasoline_pm_reference_amount = gasoline_pm_supply_chain.calculate()


# Save the interactive Sankey diagram to HTML and display it in the notebook.
gasoline_pm_sankey_path = Path.cwd() / 'D3-02_gasoline_car_pm_sankey.html'

gasoline_pm_sankey_fig = gasoline_pm_supply_chain.plot_sankey(
    gasoline_pm_sankey_df,
    width_max=1700,
    height_max=760,
    auto_width=True,
)

gasoline_pm_supply_chain.save_html(
    gasoline_pm_sankey_df,
    str(gasoline_pm_sankey_path),
    width_max=1700,
    height_max=760,
    auto_width=True,
)

print(f'Saved Sankey HTML to: {gasoline_pm_sankey_path}')
gasoline_pm_sankey_fig

## Recap

After this notebook, you should now be able to:

- load `edges` method definitions from notebook-level JSON assets
- parameterise both CH4 and N2O CFs with fixed numbers, symbolic expressions, or external Python functions
- pass one CH4 x N2O parameter grid to `EdgeLCIA` and iterate over it with repeated `evaluate_cfs()` calls
- explain why two mapped transport inventories can lead to moving 3D score surfaces when the CFs are reevaluated dynamically
- identify whether different vehicle technologies intersect under some concentration combinations
- build a country-specific external PM2.5 method that matches the supply-chain location of each edge
- use supplier `categories` to choose urban, rural, or remote intake-fraction archetypes for PM2.5
- decompose a PM2.5 CF into intake fraction and background-dependent inhaled effect factor
- call a live web API inside an external CF function and feed the returned values back into `edges`
- distinguish between fixed scenario grids and live, location-specific CF inputs
- prepare for `D3-03`, where the same reevaluation pattern is extended to explicit time-dependent and scenario-dependent tables